# Notebook 17 — Runner Characteristics and Equipment

## Bounded question

> What do the runner-level `age`, `sex` and `hg` fields represent in the source, how consistently are they populated across jurisdictions and racing types, and which values can be normalised or derived safely without inventing official eligibility, identity or equipment facts?

## Initial governed scope

The source-field governance register assigns `age`, `sex` and `hg` to the runner-level `runner_identity_and_characteristics` family.

This notebook begins with source profiling only. It does not yet assume that:

- `age` follows one universal international convention;
- runner `sex` codes have globally stable meanings;
- `hg` means headgear or equipment;
- blanks mean none rather than unknown or not supplied;
- runner characteristics prove official race eligibility.

Raw source values and SQLite storage classes remain evidence and must be preserved.


## Source lineage and read-only controls

Immutable source:

- database: `data/raw/form_2015-present/form_2015-present/raceform.db`
- table: `data`
- governed row predicate: `rowid <> 1`
- expected runner rows: `1,851,285`
- expected provisional races: `189,043`
- provisional race identity: `date + course + off`

The source database is opened in SQLite read-only mode. This notebook must not update, replace or normalise the raw table.


In [ ]:
from __future__ import annotations

import sqlite3
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)
SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"
EXPECTED_RUNNER_ROWS = 1_851_285
EXPECTED_PROVISIONAL_RACES = 189_043
RACE_KEY_COLUMNS = ["date", "course", "off"]
STUDY_FIELDS = ["age", "sex", "hg"]
GOVERNANCE_PATH = PROJECT_ROOT / "data" / "reference" / "source_field_governance.csv"

assert SOURCE_DB_PATH.exists(), SOURCE_DB_PATH
assert GOVERNANCE_PATH.exists(), GOVERNANCE_PATH

connection = sqlite3.connect(f"file:{SOURCE_DB_PATH}?mode=ro", uri=True)
connection.execute("PRAGMA query_only = ON")


## Stage 1 — Confirm governed scope and source population

This stage confirms that the live source schema still contains the three study fields, reloads their existing governance rules, and reconciles the established runner and provisional-race counts before any value profiling.


In [ ]:
source_schema = pd.read_sql_query(
    f"PRAGMA table_info({SOURCE_TABLE})",
    connection,
)

missing_fields = sorted(set(STUDY_FIELDS) - set(source_schema["name"]))
assert not missing_fields, f"Missing study fields: {missing_fields}"

field_governance = pd.read_csv(GOVERNANCE_PATH)
study_governance = field_governance.loc[
    field_governance["source_field"].isin(STUDY_FIELDS)
].sort_values("ordinal")

assert study_governance["source_field"].tolist() == STUDY_FIELDS
assert study_governance["grain"].eq("runner").all()
assert study_governance["raw_preservation"].eq("required").all()

display(study_governance)


In [ ]:
population = pd.read_sql_query(
    f'''
    SELECT
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT
            CAST(date AS TEXT) || char(31) ||
            COALESCE(CAST(course AS TEXT), '') || char(31) ||
            COALESCE(CAST(off AS TEXT), '')
        ) AS provisional_races
    FROM {SOURCE_TABLE}
    WHERE {DATA_ROW_PREDICATE}
    ''',
    connection,
)

assert int(population.loc[0, "runner_rows"]) == EXPECTED_RUNNER_ROWS
assert int(population.loc[0, "provisional_races"]) == EXPECTED_PROVISIONAL_RACES

display(population)


## Stage 2 — Raw storage and missing-state profile

Profile SQLite storage classes, SQL nulls, blank text and distinct raw values separately. No blank or sentinel is assigned a semantic meaning at this stage.


In [ ]:
storage_queries = []
for field in STUDY_FIELDS:
    storage_queries.append(
        f'''
        SELECT
            '{field}' AS source_field,
            typeof({field}) AS sqlite_storage_class,
            COUNT(*) AS runner_rows,
            SUM(CASE WHEN {field} IS NULL THEN 1 ELSE 0 END) AS sql_null_rows,
            SUM(
                CASE
                    WHEN typeof({field}) = 'text' AND trim(CAST({field} AS TEXT)) = ''
                    THEN 1 ELSE 0
                END
            ) AS blank_text_rows,
            COUNT(DISTINCT {field}) AS distinct_nonnull_raw_values
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY typeof({field})
        '''
    )

storage_profile = pd.read_sql_query(
    " UNION ALL ".join(storage_queries),
    connection,
).sort_values(["source_field", "sqlite_storage_class"])

display(storage_profile)


## Stage 3 — Initial raw vocabularies

List complete vocabularies where small enough and frequency-ranked raw values otherwise. These outputs are descriptive evidence only; labels are not yet mapped to meanings.


In [ ]:
value_profiles = {}

for field in STUDY_FIELDS:
    value_profiles[field] = pd.read_sql_query(
        f'''
        SELECT
            {field} AS raw_value,
            typeof({field}) AS sqlite_storage_class,
            COUNT(*) AS runner_rows,
            COUNT(
                DISTINCT
                CAST(date AS TEXT) || char(31) ||
                COALESCE(CAST(course AS TEXT), '') || char(31) ||
                COALESCE(CAST(off AS TEXT), '')
            ) AS provisional_races
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY {field}, typeof({field})
        ORDER BY runner_rows DESC, CAST({field} AS TEXT)
        ''',
        connection,
    )
    print(f"{field}: {len(value_profiles[field]):,} raw value/storage combinations")
    display(value_profiles[field].head(50))


## Stage 4 — Runner grain and within-race variation

A runner-level field may legitimately be constant in some races, but it must not be promoted to race grain merely because repeated values occur. This profile records within-race distinct counts without treating either variation or constancy as proof of meaning.


In [ ]:
within_race_profiles = {}

for field in STUDY_FIELDS:
    within_race_profiles[field] = pd.read_sql_query(
        f'''
        WITH race_values AS (
            SELECT
                date,
                course,
                off,
                COUNT(*) AS runner_rows,
                COUNT(DISTINCT {field}) AS distinct_nonnull_values,
                SUM(CASE WHEN {field} IS NULL THEN 1 ELSE 0 END) AS sql_null_rows,
                SUM(
                    CASE
                        WHEN typeof({field}) = 'text'
                         AND trim(CAST({field} AS TEXT)) = ''
                        THEN 1 ELSE 0
                    END
                ) AS blank_text_rows
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            GROUP BY date, course, off
        )
        SELECT
            CASE
                WHEN distinct_nonnull_values = 0 THEN 'no_nonnull_values'
                WHEN distinct_nonnull_values = 1 THEN 'one_nonnull_value'
                ELSE 'multiple_nonnull_values'
            END AS within_race_pattern,
            COUNT(*) AS provisional_races,
            SUM(runner_rows) AS runner_rows,
            MIN(distinct_nonnull_values) AS min_distinct_nonnull_values,
            MAX(distinct_nonnull_values) AS max_distinct_nonnull_values
        FROM race_values
        GROUP BY within_race_pattern
        ORDER BY within_race_pattern
        ''',
        connection,
    )
    print(field)
    display(within_race_profiles[field])


## Stage 5 — Coverage dimensions for the next investigation step

The next profiling stage should compare raw availability and vocabularies by:

- year;
- governed jurisdiction;
- race `type`;
- horse history;
- result state and source position;
- `age_band`, `sex_rest` and `comment`.

Those comparisons are intentionally not interpreted in this initial skeleton. Jurisdiction joins must reuse the governed project reference rather than reconstructing ad hoc labels.


In [ ]:
connection.close()
